# Map Algebra and Reclassification

**Series:** Raster Analysis in Python (notebook 1 of 2)
**Time:** about 30 minutes

**In this notebook you will:**
- Compare a raster with a value to make a yes/no (Boolean) raster
- Reclassify detailed land cover codes into a few groups with a lookup table
- Combine two rasters cell by cell (map algebra)
- Turn cell counts into areas and save a result as a GeoTIFF

**You should already know:** reading rasters with rasterio (Foundations <a href="https://colab.research.google.com/github/SpatialTurn/Foundations/blob/main/notebooks/04d_Raster_Data_with_rasterio.ipynb">notebook 4d</a>).

> **Before you start:** click **File > Save a copy in Drive** so your changes are kept.
> Then run each cell in order by clicking it and pressing **Shift + Enter**.

## Map algebra

A raster is a grid of numbers, so you can do arithmetic and logic on it, **cell by cell**. This is called
**map algebra**. The rule is simple: both rasters must line up exactly (same CRS, same cell size, same extent),
so that cell (row 10, column 20) in one is the same piece of ground as cell (10, 20) in the other. The two
files we use were prepared that way: EPSG:26916, 30 m cells, the same grid.

| Operation | Example | Result |
|---|---|---|
| Arithmetic | `elev * 3.281` | Elevation in feet |
| Comparison | `elev < 170` | True where land is below 170 m |
| Logic | `low & developed` | True where both are true |
| Reclassify | land cover code 21 to 24 → 2 | Detailed classes become groups |

The question for this notebook: **how much developed land in Tippecanoe County lies close to river level?**
We use "below 170 m" as a rough stand-in for land near the Wabash River. It is *not* a flood map (FEMA flood
zones are the real source for that), but it shows the method.

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from rasterio.plot import show

dem = rasterio.open("https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/tippecanoe_dem.tif")
lc = rasterio.open("https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/tippecanoe_landcover_2021.tif")

print(dem.crs == lc.crs, dem.res == lc.res, dem.bounds == lc.bounds)   # the grids line up
elev = dem.read(1, masked=True)
cover = lc.read(1, masked=True)
CELL_KM2 = 30 * 30 / 1e6                                                  # one cell is 900 m2

## Step 1: A comparison makes a Boolean raster

In [ ]:
low = elev < 170                       # True or False for every cell (masked outside the county)
print(int(low.sum()), "cells,", round(low.sum() * CELL_KM2, 1), "km2 below 170 m")

fig, ax = plt.subplots(figsize=(6, 6))
show(low.filled(False).astype("uint8"), transform=dem.transform, ax=ax, cmap="Blues")
ax.set_title("Land below 170 m (dark blue)")
plt.show()

The low land follows the Wabash valley and the lower parts of its tributaries.

## Step 2: Reclassify land cover

The National Land Cover Database (NLCD) has 16 classes in the county. For many questions a few groups are
enough. A **lookup table** maps every original code to a group, and NumPy can apply it to the whole raster in
one step: build an array whose position `i` holds the new value for code `i`.

In [ ]:
groups = {
    1: ("Water", [11, 12]),
    2: ("Developed", [21, 22, 23, 24]),
    3: ("Forest and natural", [31, 41, 42, 43, 52, 71]),
    4: ("Agriculture", [81, 82]),
    5: ("Wetlands", [90, 95]),
}
lookup = np.zeros(256, dtype="uint8")          # code -> group; 0 stays 0 (no data)
for group, (name, codes) in groups.items():
    lookup[codes] = group

grouped = np.ma.masked_equal(lookup[cover.filled(0)], 0)

for group, (name, codes) in groups.items():
    print(f"{group} {name:20s} {(grouped == group).sum() * CELL_KM2:8.1f} km2")

Always check that every original code went somewhere: a code missing from the table would silently become
0 (no data).

In [ ]:
codes_in_data = set(np.unique(cover.compressed()))
codes_in_table = {c for _, cs in groups.values() for c in cs}
print("Codes not in the table:", codes_in_data - codes_in_table)

## Step 3: Combine two rasters

Now the map algebra: developed land (group 2) **and** below 170 m.

In [ ]:
developed = grouped == 2
low_developed = low & developed
print(round(developed.sum() * CELL_KM2, 1), "km2 developed in the county")
print(round(low_developed.sum() * CELL_KM2, 1), "km2 of it below 170 m",
      f"({low_developed.sum() / developed.sum():.0%})")

fig, ax = plt.subplots(figsize=(7, 7))
show(developed.filled(False).astype("uint8"), transform=dem.transform, ax=ax, cmap="Greys", alpha=0.4)
from matplotlib.colors import ListedColormap
show(np.ma.masked_equal(low_developed.filled(False).astype("uint8"), 0), transform=dem.transform, ax=ax,
     cmap=ListedColormap(["darkorange"]))
ax.set_title("Developed land (gray) and developed land below 170 m (orange)")
plt.show()

## Step 4: Save the result

A result raster is saved with the same **profile** (CRS, transform, size) as its input, so it lines up in any
GIS. The file appears in Colab's **Files** panel and can be opened in QGIS.

In [ ]:
profile = dem.profile.copy()
profile.update(dtype="uint8", nodata=255, compress="deflate")
out = low_developed.astype("uint8").filled(255)

with rasterio.open("low_developed.tif", "w", **profile) as dst:
    dst.write(out, 1)
print("Saved low_developed.tif")

## Try it yourself

1. How many km² of **agriculture** (group 4) lie below 170 m?
2. Change the threshold to **175 m**. How much developed land is below it now? Is the answer sensitive to the
   threshold?

*Hint:* Reuse `low`, `grouped` and `CELL_KM2`; only the comparison changes.

<details>
<summary><b>Show a solution</b></summary>

```python
print(round(((grouped == 4) & low).sum() * CELL_KM2, 1), "km2 of agriculture below 170 m")
low175 = elev < 175
print(round(((grouped == 2) & low175).sum() * CELL_KM2, 1), "km2 developed below 175 m")
```

</details>

In [ ]:
# Write your answer here

## Recap

- Map algebra works cell by cell on rasters that share a grid (CRS, cell size, extent)
- A comparison (`elev < 170`) gives a Boolean raster; `&` and `|` combine them
- Reclassify with a lookup array, and check that every code is in the table
- Cells times cell area gives area; save results with the input's profile

**Next:** [02b Zonal Statistics](https://colab.research.google.com/github/SpatialTurn/DataAnalysis/blob/main/notebooks/02b_Zonal_Statistics.ipynb)